In [1]:
import gensim
from gensim.models import Word2Vec, KeyedVectors

In [2]:
import gensim.downloader as api
wv = api.load('word2vec-google-news-300')

In [3]:
wv['king'].shape

(300,)

In [80]:
import pandas as pd
messages = pd.read_csv(r'Datasets/dataset.csv')
messages.head()

,label,text
0,Spam,viiiiiiagraaaa\nonly for the ones that want to...
1,Ham,got ice thought look az original message ice o...
2,Spam,yo ur wom an ne eds an escapenumber in ch ma n...
3,Spam,start increasing your odds of success & live s...
4,Ham,author jra date escapenumber escapenumber esca...


In [81]:
import re
from nltk.corpus import stopwords

In [82]:
messages = messages[:500]

In [83]:
from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()
corpus = []
for i in range(0,len(messages)):
    review = re.sub('[^a-zA-Z]',' ',messages['text'][i])
    review = review.lower()
    review = review.split()
    review = [lemmatizer.lemmatize(word) for word in review if not word in set(stopwords.words('english'))]
    review = ' '.join(review)
    corpus.append(review)

In [84]:
from nltk import sent_tokenize
from gensim.utils import simple_preprocess

In [85]:
# simple_preprocess() # converts document into a list of lowercase tokens

In [86]:
words = []
for sent in corpus:
    sent_token = sent_tokenize(sent)
    for sent in sent_token:
        words.append(simple_preprocess(sent))

In [87]:
words[:2]

[['viiiiiiagraaaa',
  'one',
  'want',
  'make',
  'scream',
  'prodigy',
  'scrawny',
  'crow',
  'define',
  'upgrade',
  'spongy',
  'balboa',
  'dither',
  'moiseyev',
  'schumann',
  'variegate',
  'ponce',
  'bernie',
  'cox',
  'angeles',
  'impassive',
  'circulate',
  'impend',
  'miscellany',
  'chalkboard',
  'whizzing',
  'pend',
  'armenian',
  'cutlet',
  'waring',
  'makeshift',
  'fletch',
  'dispel',
  'crest',
  'cadet',
  'dovetail',
  'rapprochement',
  'gerry',
  'bayreuth',
  'selectman',
  'wilmington',
  'tuttle',
  'alchemy',
  'itt',
  'bullyboy',
  'caan'],
 ['got',
  'ice',
  'thought',
  'look',
  'az',
  'original',
  'message',
  'ice',
  'operation',
  'mailto',
  'iceoperations',
  'intcx',
  'com',
  'sent',
  'friday',
  'october',
  'escapenumber',
  'escapenumber',
  'escapenumber',
  'escapenumber',
  'pm',
  'subject',
  'escapelong',
  'amended',
  'participant',
  'agreement',
  'dear',
  'participant',
  'receiving',
  'email',
  'identified',


### Train Word2Vec from scratch

In [88]:
model = gensim.models.Word2Vec(words,vector_size=100)

In [89]:
# To get all the vocabulary
model.wv.index_to_key[:5]

['escapenumber', 'enron', 'escapelong', 'company', 'com']

In [90]:
model.corpus_count

499

In [91]:
model.epochs

5

In [92]:
model.wv.similar_by_word('good')
# These are the similar words found in given corpus.
# Not the googles pretrained corpus

[('pay', 0.9996395111083984),
 ('current', 0.999535322189331),
 ('employee', 0.9994354844093323),
 ('mean', 0.9994327425956726),
 ('done', 0.9994292259216309),
 ('american', 0.9993975758552551),
 ('analysis', 0.9993901252746582),
 ('far', 0.999387800693512),
 ('program', 0.9993544220924377),
 ('due', 0.9993416666984558)]

In [93]:
wv.similar_by_word('first')
# This is the googles model trained on 300 dimensions, and it's own WordNet

[('second', 0.7971885800361633),
 ('third', 0.6932076215744019),
 ('fourth', 0.6732367277145386),
 ('fifth', 0.6571477651596069),
 ('sixth', 0.6237859129905701),
 ('seventh', 0.591548502445221),
 ('thefirst', 0.5863957405090332),
 ('last', 0.5842218995094299),
 ('eighth', 0.5558103919029236),
 ('final', 0.5550165772438049)]

In [94]:
model.wv['first'].shape # this one was trained on only 100 dimensions

(100,)

In [95]:
# In normal Word2Vec
# each word will have 100 dimensions: each dimension explaining the similarity of it against the word.
# but in a given sentence there will be multiple words,leading to n * 100 vectors.
# Instead, for a given sentence we can take a average across all words for a given dimension.
# with this we can reduce the vector size of the sentence to 100
# this is called AvgWord2Vec
words[0] # This is one sentence, each of these words will have 100 dimension vectors.

['viiiiiiagraaaa',
 'one',
 'want',
 'make',
 'scream',
 'prodigy',
 'scrawny',
 'crow',
 'define',
 'upgrade',
 'spongy',
 'balboa',
 'dither',
 'moiseyev',
 'schumann',
 'variegate',
 'ponce',
 'bernie',
 'cox',
 'angeles',
 'impassive',
 'circulate',
 'impend',
 'miscellany',
 'chalkboard',
 'whizzing',
 'pend',
 'armenian',
 'cutlet',
 'waring',
 'makeshift',
 'fletch',
 'dispel',
 'crest',
 'cadet',
 'dovetail',
 'rapprochement',
 'gerry',
 'bayreuth',
 'selectman',
 'wilmington',
 'tuttle',
 'alchemy',
 'itt',
 'bullyboy',
 'caan']

In [96]:
import numpy as np

In [97]:
def avg_word2vec(doc):
    vectors = [model.wv[word] for word in doc if word in model.wv.index_to_key]
    if len(vectors) == 0:
        # If no sentence is present return the 0,0,0--0 vector
        return np.zeros(model.vector_size)
    return np.mean(vectors, axis=0)

In [98]:
pip install tqdm

Note: you may need to restart the kernel to use updated packages.


In [99]:
from tqdm import tqdm
# It's just used to see the progress

In [100]:
X=[]
for i in tqdm(range(len(words))):
    X.append(avg_word2vec(words[i]))

100%|██████████| 499/499 [00:03<00:00, 127.80it/s]


In [101]:
shapes = set(np.shape(x) for x in X)
print(shapes)

{(100,)}


In [102]:
len(X)

499

In [103]:
X=np.array(X)

In [104]:
X[0].shape # Each sentence will have 100 dimensions

(100,)

In [105]:
X.shape

(499, 100)

In [106]:
# Dependent Feature
y = pd.get_dummies(messages['label'])
y

,Ham,Spam
0,False,True
1,True,False
2,False,True
3,False,True
4,True,False
...,...,...
495,True,False
496,False,True
497,True,False
498,True,False


In [107]:
y = y.iloc[:,1].values #Taking either one column will do the job for considering o/p

In [108]:
X.shape

(499, 100)

In [115]:
y = y[:499]

In [116]:
y.shape

(499,)

In [117]:
# Train Test Split
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2)